# Möbius homology calculator — Colab version

This notebook is self-contained and should run directly in Google Colab.

It computes the chain complex from a perfect-matching ribbon graph using the Möbius Frobenius algebra
\[
V=\mathbb Q[a,b]/(a^n, b^3-ab).
\]

The implementation tracks the actual smoothing circles/components, uses genuine cube edges only, applies \(\mu\), \(\Delta\), and \(m\) to the correct tensor factors, and uses exact rational linear algebra via `sympy`.

Run the cells from top to bottom. The final cells contain editable examples.


In [ ]:
# Colab normally already has sympy. This is harmless if it is already installed.
try:
    import sympy as sp
    print('SymPy version:', sp.__version__)
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sympy'])
    import sympy as sp
    print('SymPy version:', sp.__version__)


## Core code

You usually do not need to edit this cell. Edit the examples below instead.

In [ ]:
"""
Reliable exact computation of the Möbius-strip Frobenius-algebra homology
for the ribbon-graph input format used in Möbius_Frobenius_alg.ipynb.

Main improvements over the original notebook:
  * genuine cube edges only;
  * each smoothing stores the actual circles/components, not just their number;
  * merge/split/self-crossing maps are applied to the correct tensor factors;
  * all linear algebra is exact over QQ, via sympy Rational matrices;
  * top homology is computed as ker(d_top)/im(d_prev), not by a shortcut.

Input format:
  pairs[i] = [left_neighbor, right_neighbor] away from the perfect matching.
  matching = [[v,w], ...] is the list of perfect-matching edges.

Example:
  pairs = [[2,3], [3,2], [1,0], [0,1]]
  matching = [[0,1], [2,3]]
  MH = MobiusHomology(pairs, matching, n=1)
  print(MH.state_circle_counts())
  print(MH.homology_dimensions())
  print(MH.poincare_polynomial())
"""

from __future__ import annotations

from dataclasses import dataclass
from fractions import Fraction
from itertools import combinations, product
from typing import Dict, Iterable, List, Sequence, Tuple

import sympy as sp

Endpoint = Tuple[int, int]       # (vertex, side), side is -1 or +1
Edge = Tuple[Endpoint, Endpoint]
State = Tuple[int, ...]
BasisElt = Tuple[int, int]       # a^i b^j with 0 <= i < n and j in {0,1,2}
TensorElt = Tuple[int, ...]      # tuple of basis indices


def _canonical_edge(u: Endpoint, v: Endpoint) -> Edge:
    return (u, v) if u <= v else (v, u)


def _binary_states(num_bits: int, weight: int) -> List[State]:
    out: List[State] = []
    for ones in combinations(range(num_bits), weight):
        arr = [0] * num_bits
        for i in ones:
            arr[i] = 1
        out.append(tuple(arr))
    return out


def _cube_sign(source: State, changed_position: int) -> int:
    """Standard cube sign: (-1)^(number of 1s before the changed coordinate)."""
    return -1 if sum(source[:changed_position]) % 2 else 1


@dataclass(frozen=True)
class SmoothingData:
    state: State
    circles: Tuple[frozenset[Endpoint], ...]

    @property
    def k(self) -> int:
        return len(self.circles)


class MobiusAlgebra:
    """The algebra QQ[a,b]/(a^n, b^3-a*b) with the Möbius Frobenius structure."""

    def __init__(self, n: int = 1):
        if n < 1:
            raise ValueError("n must be positive")
        self.n = int(n)
        self.basis: List[BasisElt] = [(i, j) for i in range(self.n) for j in range(3)]
        self.index: Dict[BasisElt, int] = {b: i for i, b in enumerate(self.basis)}
        self.dim = len(self.basis)
        self._pairing_inv = None
        self._delta_one = None

    def degree_of_index(self, idx: int) -> int:
        i, j = self.basis[idx]
        return (2 * i + j) % self.n

    def tensor_degree(self, tensor: TensorElt) -> int:
        return sum(self.degree_of_index(i) for i in tensor) % self.n

    def multiply_basis(self, p: int, q: int) -> Dict[int, Fraction]:
        """Product of two basis elements, returned sparsely."""
        ai, bj = self.basis[p]
        ak, bl = self.basis[q]
        a_exp = ai + ak
        b_exp = bj + bl
        while b_exp >= 3:
            # b^3 = a*b, so b^r = a*b^(r-2) for r >= 3.
            a_exp += 1
            b_exp -= 2
        if a_exp >= self.n:
            return {}
        return {self.index[(a_exp, b_exp)]: Fraction(1)}

    def mobius_element_index(self) -> int:
        if self.n % 2 == 1:
            return self.index[((self.n - 1) // 2, 1)]
        return self.index[((self.n - 2) // 2, 2)]

    def mobius_map_on_basis(self, p: int) -> Dict[int, Fraction]:
        return self.multiply_basis(self.mobius_element_index(), p)

    def epsilon_on_basis(self, p: int) -> Fraction:
        return Fraction(3 * self.n) if p == self.dim - 1 else Fraction(0)

    def pairing_matrix(self) -> sp.Matrix:
        M = sp.zeros(self.dim, self.dim)
        for i in range(self.dim):
            for j in range(self.dim):
                s = Fraction(0)
                for k, c in self.multiply_basis(i, j).items():
                    s += c * self.epsilon_on_basis(k)
                M[i, j] = sp.Rational(s.numerator, s.denominator)
        return M

    def pairing_inverse(self) -> sp.Matrix:
        if self._pairing_inv is None:
            self._pairing_inv = self.pairing_matrix().inv()
        return self._pairing_inv

    def delta_one(self) -> Dict[Tuple[int, int], Fraction]:
        """Delta(1) = sum_{i,j} (pairing^{-1})_{i,j} e_i tensor e_j."""
        if self._delta_one is None:
            inv = self.pairing_inverse()
            d: Dict[Tuple[int, int], Fraction] = {}
            for i in range(self.dim):
                for j in range(self.dim):
                    val = inv[i, j]
                    if val:
                        d[(i, j)] = Fraction(int(val.p), int(val.q))
            self._delta_one = d
        return self._delta_one

    def delta_on_basis(self, p: int) -> Dict[Tuple[int, int], Fraction]:
        """Delta(e_p) = (e_p tensor 1) Delta(1), for this commutative Frobenius algebra."""
        out: Dict[Tuple[int, int], Fraction] = {}
        for (i, j), c in self.delta_one().items():
            for k, ck in self.multiply_basis(p, i).items():
                out[(k, j)] = out.get((k, j), Fraction(0)) + c * ck
        return {k: v for k, v in out.items() if v}


class MobiusHomology:
    def __init__(self, pairs: Sequence[Sequence[int]], matching: Sequence[Sequence[int]], n: int = 1):
        self.pairs = [tuple(map(int, p)) for p in pairs]
        self.matching = [tuple(map(int, m)) for m in matching]
        self.num_matchings = len(self.matching)
        self.alg = MobiusAlgebra(n=n)
        self._base_edges = self._non_matching_edges()
        self._smoothing_cache: Dict[State, SmoothingData] = {}
        self._tensor_basis_cache: Dict[Tuple[int, int | None], List[TensorElt]] = {}

    # ----- graph and smoothing data -----

    def _non_matching_edges(self) -> List[Edge]:
        """Same graph convention as the original notebook, but normalized."""
        raw: List[Edge] = []
        pairs = self.pairs
        for i in range(len(pairs)):
            left, right = pairs[i]
            if left != right:
                if pairs[left][0] == i:
                    raw.append(_canonical_edge((i, -1), (left, -1)))
                elif pairs[right][1] == i:
                    raw.append(_canonical_edge((i, 1), (left, 1)))
                elif pairs[left][1] == i:
                    raw.append(_canonical_edge((i, -1), (left, 1)))
                else:
                    raw.append(_canonical_edge((i, 1), (left, -1)))
            else:
                raw.append(_canonical_edge((i, -1), (left, -1)))
                raw.append(_canonical_edge((i, 1), (left, 1)))
        # remove duplicates while preserving order
        seen = set()
        out = []
        for e in raw:
            if e not in seen:
                seen.add(e)
                out.append(e)
        return out

    def smoothing_edges(self, state: State) -> List[Edge]:
        if len(state) != self.num_matchings:
            raise ValueError("state length does not match number of matching edges")
        edges = list(self._base_edges)
        for bit, (u, v) in zip(state, self.matching):
            if bit == 0:
                edges.append(_canonical_edge((u, 1), (v, 1)))
                edges.append(_canonical_edge((u, -1), (v, -1)))
            elif bit == 1:
                edges.append(_canonical_edge((u, 1), (v, -1)))
                edges.append(_canonical_edge((u, -1), (v, 1)))
            else:
                raise ValueError("states must be 0/1")
        return edges

    def smoothing_data(self, state: State) -> SmoothingData:
        state = tuple(state)
        if state in self._smoothing_cache:
            return self._smoothing_cache[state]

        adj: Dict[Endpoint, List[Endpoint]] = {}
        for u, v in self.smoothing_edges(state):
            adj.setdefault(u, []).append(v)
            adj.setdefault(v, []).append(u)

        # In a closed smoothing, every half-edge has valence 2.
        bad = {u: vs for u, vs in adj.items() if len(vs) != 2}
        if bad:
            raise ValueError(f"smoothing {state} is not a disjoint union of circles; bad endpoints: {bad}")

        seen = set()
        circles: List[frozenset[Endpoint]] = []
        for start in sorted(adj):
            if start in seen:
                continue
            comp = set()
            stack = [start]
            seen.add(start)
            while stack:
                u = stack.pop()
                comp.add(u)
                for v in adj[u]:
                    if v not in seen:
                        seen.add(v)
                        stack.append(v)
            circles.append(frozenset(comp))
        circles.sort(key=lambda c: sorted(c))
        data = SmoothingData(state=state, circles=tuple(circles))
        self._smoothing_cache[state] = data
        return data

    def states_by_homological_degree(self) -> List[List[State]]:
        return [_binary_states(self.num_matchings, i) for i in range(self.num_matchings + 1)]

    def state_circle_counts(self) -> List[List[int]]:
        return [[self.smoothing_data(s).k for s in states] for states in self.states_by_homological_degree()]

    def cube_edges(self, degree: int) -> List[Tuple[State, State, int, int]]:
        """Edges C^degree -> C^(degree+1). Returns source, target, changed coordinate, sign."""
        out = []
        for s in _binary_states(self.num_matchings, degree):
            for pos, bit in enumerate(s):
                if bit == 0:
                    t = list(s)
                    t[pos] = 1
                    t = tuple(t)
                    out.append((s, t, pos, _cube_sign(s, pos)))
        return out

    # ----- tensor bases -----

    def tensor_basis(self, k: int, internal_degree: int | None = None) -> List[TensorElt]:
        key = (k, internal_degree)
        if key in self._tensor_basis_cache:
            return self._tensor_basis_cache[key]
        all_tuples = list(product(range(self.alg.dim), repeat=k))
        if internal_degree is not None:
            all_tuples = [v for v in all_tuples if self.alg.tensor_degree(v) == internal_degree % self.alg.n]
        self._tensor_basis_cache[key] = all_tuples
        return all_tuples

    def chain_basis(self, degree: int, internal_degree: int | None = None) -> List[Tuple[State, TensorElt]]:
        out: List[Tuple[State, TensorElt]] = []
        for state in _binary_states(self.num_matchings, degree):
            k = self.smoothing_data(state).k
            for ten in self.tensor_basis(k, internal_degree):
                out.append((state, ten))
        return out

    # ----- local maps on actual tensor factors -----

    def _component_indices_touching(self, data: SmoothingData, endpoints: Iterable[Endpoint]) -> List[int]:
        endpoints = set(endpoints)
        return [i for i, c in enumerate(data.circles) if c & endpoints]

    def _edge_map_on_tensor(self, source: State, target: State, changed_pos: int, x: TensorElt) -> Dict[TensorElt, Fraction]:
        sd = self.smoothing_data(source)
        td = self.smoothing_data(target)
        u, v = self.matching[changed_pos]
        touched = [(u, -1), (u, 1), (v, -1), (v, 1)]
        src_touch = self._component_indices_touching(sd, touched)
        tgt_touch = self._component_indices_touching(td, touched)

        # Components not involved in the change are identified by the same endpoint set.
        source_comp_to_target: Dict[int, int] = {}
        target_comp_to_source: Dict[int, int] = {}
        for i, c in enumerate(sd.circles):
            for j, d in enumerate(td.circles):
                if c == d:
                    source_comp_to_target[i] = j
                    target_comp_to_source[j] = i

        out: Dict[TensorElt, Fraction] = {}

        if sd.k == td.k + 1:
            # merge: two source circles -> one target circle
            if len(src_touch) != 2 or len(tgt_touch) != 1:
                raise ValueError(f"unexpected merge data {source}->{target}: {src_touch}, {tgt_touch}")
            a, b = src_touch
            tgt = tgt_touch[0]
            local = self.alg.multiply_basis(x[a], x[b])
            for prod_idx, coeff in local.items():
                y = [None] * td.k
                y[tgt] = prod_idx
                for i in range(sd.k):
                    if i not in (a, b):
                        y[source_comp_to_target[i]] = x[i]
                out[tuple(y)] = out.get(tuple(y), Fraction(0)) + coeff

        elif sd.k + 1 == td.k:
            # split: one source circle -> two target circles
            if len(src_touch) != 1 or len(tgt_touch) != 2:
                raise ValueError(f"unexpected split data {source}->{target}: {src_touch}, {tgt_touch}")
            src = src_touch[0]
            t0, t1 = tgt_touch
            local = self.alg.delta_on_basis(x[src])
            for (left, right), coeff in local.items():
                y = [None] * td.k
                y[t0] = left
                y[t1] = right
                for i in range(sd.k):
                    if i != src:
                        y[source_comp_to_target[i]] = x[i]
                out[tuple(y)] = out.get(tuple(y), Fraction(0)) + coeff

        elif sd.k == td.k:
            # self-crossing/Möbius: one source circle -> one target circle
            if len(src_touch) != 1 or len(tgt_touch) != 1:
                raise ValueError(f"unexpected Mobius data {source}->{target}: {src_touch}, {tgt_touch}")
            src = src_touch[0]
            tgt = tgt_touch[0]
            local = self.alg.mobius_map_on_basis(x[src])
            for mob_idx, coeff in local.items():
                y = [None] * td.k
                y[tgt] = mob_idx
                for i in range(sd.k):
                    if i != src:
                        y[source_comp_to_target[i]] = x[i]
                out[tuple(y)] = out.get(tuple(y), Fraction(0)) + coeff
        else:
            raise ValueError(f"circle count changed by more than one: {source}->{target}")

        # Defensive check: no tensor position was left unfilled.
        clean: Dict[TensorElt, Fraction] = {}
        for y, c in out.items():
            if any(z is None for z in y):
                raise RuntimeError(f"internal error: incomplete output tensor {y} for {source}->{target}")
            clean[y] = clean.get(y, Fraction(0)) + c
        return {k: v for k, v in clean.items() if v}

    def differential_matrix(self, degree: int, internal_degree: int | None = None) -> sp.Matrix:
        """Matrix of d^degree: C^degree -> C^(degree+1)."""
        if degree < 0 or degree >= self.num_matchings:
            return sp.zeros(0, 0)
        src_basis = self.chain_basis(degree, internal_degree)
        tgt_basis = self.chain_basis(degree + 1, internal_degree)
        src_index = {b: i for i, b in enumerate(src_basis)}
        tgt_index = {b: i for i, b in enumerate(tgt_basis)}
        D = sp.MutableSparseMatrix(len(tgt_basis), len(src_basis), {})

        for source, target, changed_pos, sign in self.cube_edges(degree):
            for x in self.tensor_basis(self.smoothing_data(source).k, internal_degree):
                col = src_index[(source, x)]
                for y, coeff in self._edge_map_on_tensor(source, target, changed_pos, x).items():
                    if internal_degree is not None and self.alg.tensor_degree(y) != internal_degree % self.alg.n:
                        raise RuntimeError("local map did not preserve internal degree")
                    row = tgt_index[(target, y)]
                    val = Fraction(sign) * coeff
                    D[row, col] += sp.Rational(val.numerator, val.denominator)
        return sp.Matrix(D)

    # ----- homology -----

    def homology_dimensions(self, internal_degree: int | None = None) -> List[int]:
        dims = []
        prev_rank = 0
        for i in range(self.num_matchings + 1):
            chain_dim = len(self.chain_basis(i, internal_degree))
            if i < self.num_matchings:
                rank_next = self.differential_matrix(i, internal_degree).rank()
            else:
                rank_next = 0
            dims.append(int(chain_dim - rank_next - prev_rank))
            prev_rank = rank_next
        return dims

    def homology_dimensions_by_internal_degree(self) -> Dict[int, List[int]]:
        return {d: self.homology_dimensions(d) for d in range(self.alg.n)}

    def check_d_squared(self, internal_degree: int | None = None) -> bool:
        for i in range(self.num_matchings - 1):
            if self.differential_matrix(i + 1, internal_degree) * self.differential_matrix(i, internal_degree) != sp.zeros(
                len(self.chain_basis(i + 2, internal_degree)), len(self.chain_basis(i, internal_degree))
            ):
                return False
        return True

    def poincare_polynomial(self, variable_t: str = "t", variable_q: str = "q") -> str:
        t, q = sp.symbols(f"{variable_t} {variable_q}")
        poly = 0
        for d, hs in self.homology_dimensions_by_internal_degree().items():
            for i, dim in enumerate(hs):
                poly += dim * (q ** d) * (t ** i)
        return str(sp.expand(poly))


if __name__ == "__main__":
    easy_pairs = [[1, 1], [0, 0]]
    easy_matching = [[0, 1]]

    next_pairs = [[2, 3], [3, 2], [1, 0], [0, 1]]
    next_matching = [[0, 1], [2, 3]]

    for name, pairs, matching in [
        ("easy", easy_pairs, easy_matching),
        ("next", next_pairs, next_matching),
    ]:
        print("\nExample:", name)
        MH = MobiusHomology(pairs, matching, n=1)
        print("circle counts:", MH.state_circle_counts())
        print("d^2=0:", MH.check_d_squared())
        print("homology dimensions:", MH.homology_dimensions())
        print("Poincare polynomial:", MH.poincare_polynomial())


## Test example from the original notebook

This should print circle counts `[[2], [1, 1], [1]]`, verify `d^2=0`, and give homology dimensions `[6, 1, 1]`, hence \(t^2+t+6\).

In [ ]:
next_pairs = [[2, 3], [3, 2], [1, 0], [0, 1]]
next_matching = [[0, 1], [2, 3]]

MH = MobiusHomology(next_pairs, next_matching, n=1)
print('circle counts:', MH.state_circle_counts())
print('d^2 = 0:', MH.check_d_squared())
print('homology dimensions:', MH.homology_dimensions())
print('Poincare polynomial:', MH.poincare_polynomial())


## Another tiny example

In [ ]:
easy_pairs = [[1, 1], [0, 0]]
easy_matching = [[0, 1]]

MH = MobiusHomology(easy_pairs, easy_matching, n=1)
print('circle counts:', MH.state_circle_counts())
print('d^2 = 0:', MH.check_d_squared())
print('homology dimensions:', MH.homology_dimensions())
print('Poincare polynomial:', MH.poincare_polynomial())


## Edit this cell for your own graph

Input format is the same as in the old notebook:

```python
pairs[i] = [left_neighbor, right_neighbor]
matching = [[u, v], ...]
```

Set `n=1` for the example algebra \(\mathbb Q[y]/(y^3)\).

In [ ]:
# Replace these with your graph.
pairs = [[2, 3], [3, 2], [1, 0], [0, 1]]
matching = [[0, 1], [2, 3]]
n = 1

MH = MobiusHomology(pairs, matching, n=n)
print('circle counts by homological degree:', MH.state_circle_counts())
print('d^2 = 0:', MH.check_d_squared())
print('homology dimensions by homological degree:', MH.homology_dimensions())
print('homology dimensions by internal degree:', MH.homology_dimensions_by_internal_degree())
print('Poincare polynomial:', MH.poincare_polynomial())


## Optional: inspect a differential matrix

For example, `degree=0` prints the matrix of \(d^0:C^0	o C^1\).

In [ ]:
degree = 0
D = MH.differential_matrix(degree)
print('shape:', D.shape)
D
